# 관계와 근거 연결을 확인하는 합성 예제

카드 정보나 실제 PDF를 사용하지 않습니다. 이전 문제를 **합성으로 재구성**하고, 현재 구현의 중복 관계 재연결과 자기 연결 근거 보존을 확인합니다. 실제 데이터와 챗봇 사용자 영향은 이 노트북으로 확인할 수 없습니다.

## 1. 준비

저장소 루트 또는 그 하위 폴더에서 실행합니다. 기존 파이프라인 함수와 사용자 점검 함수를 직접 호출합니다.

In [ ]:
from pathlib import Path
import sys
from copy import deepcopy

candidates = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
root = next((p for p in candidates if (p / "src/lee_file_pipeline/evidence_audit.py").is_file() and (p / "src/card_benefit/payment_card/file_pipeline").is_dir()), None)
if root is None:
    raise RuntimeError("저장소 루트 또는 그 하위 폴더에서 노트북을 실행하세요.")
for path in (root / "src", root / "src/card_benefit"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from lee_file_pipeline.evidence_audit import audit_graph
from payment_card.file_pipeline.chat.retrieval.entity_dedupe import dedupe_subgraph
print("합성 예제 준비 완료")

## 2. 이전 문제를 합성으로 재구성

두 관계 중 `e1`만 남았는데 근거는 `e1`, `e2`를 가리키는 상태를 직접 구성합니다. 이전 버전을 실행하는 것은 아닙니다.

In [ ]:
graph = {
    "nodes": [{"id": "a"}, {"id": "b"}, {"id": "t"}],
    "edges": [
        {"_fp_id": "e1", "source": "a", "target": "t", "type": "OFFERS"},
        {"_fp_id": "e2", "source": "b", "target": "t", "type": "OFFERS"},
    ],
    "evidence": [{"edge_id": "e1", "source": "synthetic-1"}, {"edge_id": "e2", "source": "synthetic-2"}],
    "seeds": ["a", "b"],
}
before = deepcopy(graph)
before["edges"] = [before["edges"][0]]
before_report = audit_graph(before)
assert before_report["dangling_edge_ids"] == ["e2"]
print("재구성한 끊어진 참조:", before_report["dangling_edge_ids"])

## 3. 현재 구현으로 관계 병합

`b`를 `a`로 합치고 현재 함수를 호출합니다. 근거 두 행이 유지되고 변경된 행에 원래 관계 ID가 남는지 확인합니다.

In [ ]:
after = dedupe_subgraph(graph, {"b": "a"})
after_report = audit_graph(after)
assert after_report["status"] == "ok"
assert len(after["evidence"]) == 2
assert any(row.get("_dedupe_original_edge_id") == "e2" for row in after["evidence"])
print("남은 관계:", [edge.get("_fp_id") for edge in after["edges"]])
print("근거 참조와 원래 ID:", [(row.get("edge_id"), row.get("_dedupe_original_edge_id")) for row in after["evidence"]])
print("연결 점검:", after_report["status"])

## 4. 자기 연결로 제거된 관계의 근거

대표 관계가 없는 자기 연결의 근거는 활성 관계에 임의로 붙이지 않고 사유와 함께 별도 보존합니다.

In [ ]:
loop_graph = {"nodes": [{"id": "a"}, {"id": "b"}],
              "edges": [{"_fp_id": "loop-1", "source": "a", "target": "b", "type": "OFFERS"}],
              "evidence": [{"edge_id": "loop-1", "source": "synthetic-loop"}]}
loop_result = dedupe_subgraph(loop_graph, {"b": "a"})
loop_report = audit_graph(loop_result)
assert not loop_result["edges"]
assert loop_result["unresolved_edge_evidence"][0]["reason"] == "removed_self_loop"
assert loop_report["status"] == "findings"
print("별도 보존 건수:", loop_report["unresolved_edge_evidence_count"])
print("사유:", loop_result["unresolved_edge_evidence"][0]["reason"])

## 확인 범위와 피드백 질문

- 이 노트북은 합성 입력에서의 코드 동작만 보여 줍니다. 실제 PDF, DB, 평가 데이터 또는 챗봇 화면을 검증하지 않았습니다.
- 자기 연결 근거의 향후 소비 방식과 추가된 진단 CSV 열의 기존 소비자 호환성은 미확인입니다.
- 피드백: 대표 관계에 재연결하면서 원래 ID를 남기는 방식과, 자기 연결 근거를 별도 보존하는 방식이 적절한가요?